# BullsEye: ColorBench Evaluation Pipeline
Model loads once and is reused across all phases.

## ⚙️ Step 1 — Config

In [ ]:
# HuggingFace token — add HF_TOKEN to Colab Secrets (🔑 left sidebar)
from google.colab import userdata
from huggingface_hub import login
import os
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
login(token=os.environ['HF_TOKEN'], add_to_git_credential=False)

# Model: 3B (~3.5 GB VRAM) or 7B (~7.5 GB VRAM)
MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"

# Samples to evaluate:
# 100  → quick test   (~10 min per eval phase)
# 1448 → full dataset (~50 min per eval phase, matches paper)
NUM_SAMPLES = 1448

# Results saved here (persists across sessions)
OUTPUT_DIR = "/content/drive/MyDrive/bullseye_results"

print(f"Model   : {MODEL_ID}")
print(f"Samples : {NUM_SAMPLES}")
print(f"Output  : {OUTPUT_DIR}")

## 💾 Step 2 — Mount Google Drive

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Ready. Results → {OUTPUT_DIR}")

## 🔧 Step 3 — Clone Repo & Install Dependencies

In [ ]:
import os
REPO_URL = "https://github.com/tanzim12911/cse465-project.git"
if not os.path.exists("cse465-project"):
    !git clone $REPO_URL
else:
    !git -C cse465-project pull
os.chdir("cse465-project")
import sys; sys.path.insert(0, 'src')
print("cwd:", os.getcwd())

In [ ]:
# Colab already has torch, transformers, datasets, opencv, sklearn, Pillow, tqdm
!pip install -q bitsandbytes qwen-vl-utils

import torch
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU :", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory/1e9, 1), "GB")

## ⬇️ Step 4 — Download Weights & Dataset

In [ ]:
from huggingface_hub import snapshot_download
import time
print(f"Downloading {MODEL_ID} weights...")
t = time.time()
snapshot_download(
    repo_id=MODEL_ID,
    ignore_patterns=["*.msgpack", "*.h5", "flax_model*", "tf_model*"]
)
print(f"Done in {(time.time()-t)/60:.1f} min")

In [ ]:
import subprocess

# Download dataset (full if NUM_SAMPLES is None, otherwise a subset)
cmd = ['python', 'src/data/download_colorbench.py']
if NUM_SAMPLES is not None:
    cmd += ['--max_samples', str(NUM_SAMPLES)]
subprocess.run(cmd, check=True)

# Create text-only variant for blind eval
subprocess.run(['python', 'src/data/image_stripper.py'], check=True)

# Generate taxonomy map for dispatcher
subprocess.run(['python', 'src/probing/taxonomy_cluster.py'], check=True)

## 🤖 Step 5 — Load Model
> Run **once per session**. All eval cells reuse this model — no reload.

In [ ]:
import torch, time
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration, BitsAndBytesConfig

print(f"Loading {MODEL_ID} in 4-bit...")
t = time.time()
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    device_map="auto",
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
    )
)
processor = AutoProcessor.from_pretrained(MODEL_ID)
model.eval()
print(f"Ready in {(time.time()-t)/60:.1f} min  |  VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB")

## Phase 0 — Baseline Evaluation (Zero-Shot)

In [ ]:
from datasets import load_from_disk
from eval.baseline_eval import run_zero_shot_evaluation

dataset   = load_from_disk("./data/colorbench")
eval_data = dataset[list(dataset.keys())[0]]

baseline_acc, baseline_stats = run_zero_shot_evaluation(
    model, processor, eval_data,
    output_dir=OUTPUT_DIR,
    num_samples=NUM_SAMPLES
)

## Phase 1 — Blind Baseline (Text-Only)

In [ ]:
from eval.blind_eval import run_blind_evaluation

stripped   = load_from_disk("./data/colorbench_stripped")
blind_data = stripped[list(stripped.keys())[0]]

blind_acc, blind_stats = run_blind_evaluation(
    model, processor, blind_data,
    output_dir=OUTPUT_DIR,
    num_samples=NUM_SAMPLES
)

## Phase 2 — BullsEye Evaluation

In [ ]:
from eval.bullseye_eval import run_bullseye_evaluation

bullseye_acc, bullseye_stats = run_bullseye_evaluation(
    model, processor, eval_data,
    output_dir=OUTPUT_DIR,
    num_samples=NUM_SAMPLES
)

## Results — Baseline vs BullsEye

In [ ]:
import json, os

def load_results(filepath):
    stats = {}
    if not os.path.exists(filepath):
        print(f"Not found: {filepath}"); return stats
    with open(filepath) as f:
        for line in f:
            d = json.loads(line)
            t = d.get('task', 'Unknown')
            stats.setdefault(t, {'correct': 0, 'total': 0})
            stats[t]['total'] += 1
            if d.get('correct'): stats[t]['correct'] += 1
    return stats

B  = load_results(os.path.join(OUTPUT_DIR, 'baseline_results.jsonl'))
BE = load_results(os.path.join(OUTPUT_DIR, 'bullseye_results.jsonl'))
all_tasks = sorted(set(list(B.keys()) + list(BE.keys())))

print(f"\n{'Task':<26} {'Baseline':>9} {'BullsEye':>9} {'Delta':>7}")
print("-" * 56)
bt, bc, et, ec = 0, 0, 0, 0
for task in all_tasks:
    b  = B.get(task,  {'correct': 0, 'total': 0})
    be = BE.get(task, {'correct': 0, 'total': 0})
    ba  = b['correct']  / b['total']  * 100 if b['total']  else 0
    bea = be['correct'] / be['total'] * 100 if be['total'] else 0
    d   = bea - ba
    sign = '+' if d >= 0 else ''
    print(f"{task:<26} {ba:>8.1f}% {bea:>8.1f}% {sign}{d:>6.1f}%")
    bt+=b['total'];  bc+=b['correct']
    et+=be['total']; ec+=be['correct']
print("-" * 56)
bo  = bc/bt*100  if bt else 0
beo = ec/et*100  if et else 0
do  = beo - bo
sign = '+' if do >= 0 else ''
print(f"{'OVERALL':<26} {bo:>8.1f}% {beo:>8.1f}% {sign}{do:>6.1f}%")
print(f"\nModel: {MODEL_ID}  |  Samples: {NUM_SAMPLES}")

## 📸 Custom Image Demo

1. Upload your 5 images below
2. Edit `CUSTOM_SAMPLES` in `src/eval/custom_demo.py`
3. Run the demo cell

| Task | Branch |
|---|---|
| Color Counting, Color Comparison, Color Proportion, Object Counting, Color Blindness, Color Recognition, Object Recognition | Reasoning (CoT) |
| Color Extraction | Extraction (HSV) |
| Color Illusion, Color Mimicry | Suppression (grayscale) |
| Color Robustness | Normalization |

In [ ]:
import os
from google.colab import files
os.makedirs('./demo_images', exist_ok=True)
uploaded = files.upload()
for fname, data in uploaded.items():
    open(f'./demo_images/{fname}', 'wb').write(data)
    print(f"Saved: ./demo_images/{fname}")

In [ ]:
import importlib
import eval.custom_demo as cd
importlib.reload(cd)
cd.run_custom_demo_inline(model, processor, OUTPUT_DIR)